# Part 1 — Imports

In [78]:
from pathlib import Path
import torch

from dmpbridge.pdf.page_image_converter import convert_pdf_to_images
from dmpbridge.vision.qwen_structure_detector import detect_structure_from_images
from dmpbridge.vision.qwen_postprocessor import save_qwen_structured_blocks
from dmpbridge.processing.structure_json_builder import save_narrative_json
from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure

# Part 2 — Check GPU

In [79]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

Torch version: 2.6.0+cu124
CUDA available: True
GPU count: 1
0 NVIDIA GeForce RTX 3090


# Part 3 — Paths

In [80]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample10.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

qwen_output_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}.json"
qwen_structured_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}_structured_blocks.json"
qwen_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_qwen.json"

print("Project root:", project_root)
print("PDF path:", pdf_path)
print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

Project root: c:\Users\Nahid\dmpbridge
PDF path: c:\Users\Nahid\dmpbridge\data\raw_pdfs\sample10.pdf
PDF exists: True
Skeleton exists: True


# Part 4 — Convert PDF to page images

In [81]:
image_paths = convert_pdf_to_images(pdf_path, dpi=120)

print("Number of page images:", len(image_paths))

for p in image_paths:
    print(p, p.exists())

[2026-05-08 14:09:29] Converting PDF pages to images: sample10.pdf
[2026-05-08 14:09:29] Saved 2 page images to: C:\Users\Nahid\dmpbridge\data\page_images\sample10
Number of page images: 2
C:\Users\Nahid\dmpbridge\data\page_images\sample10\page_1.png True
C:\Users\Nahid\dmpbridge\data\page_images\sample10\page_2.png True


# Part 5 — Run Qwen2-VL structure detection

In [82]:
qwen_results = detect_structure_from_images(
    image_paths=image_paths,
    output_path=qwen_output_path
)

print("Saved Qwen output:", qwen_output_path.exists())
print("Qwen output path:", qwen_output_path)

qwen_results

[2026-05-08 14:09:29] Loading Qwen2-VL model: Qwen/Qwen2-VL-7B-Instruct


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

[2026-05-08 14:10:06] Running Qwen2-VL on page 1: C:\Users\Nahid\dmpbridge\data\page_images\sample10\page_1.png
[2026-05-08 14:11:06] Running Qwen2-VL on page 2: C:\Users\Nahid\dmpbridge\data\page_images\sample10\page_2.png
[2026-05-08 14:16:16] Saved Qwen structure output: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample10.json
Saved Qwen output: True
Qwen output path: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample10.json


[{'document_title': None,
  'sections': [{'title': 'Policy and Practice',
    'subsections': [{'title': 'Scope'},
     {'title': 'Data and Metadata Format and Contents'}]}],
  'page': 1},
 {'document_title': None,
  'sections': [{'title': 'Accessibility and Data Protection',
    'subsections': [{'title': 'Materials will be searchable through a number of databases and search engines, including Google and Google Scholar. As noted in Section 2 (Scope), data will be made available only after appropriate steps have been taken to protect intellectual property. Confidential material will be handled according to policies and protocols for human subjects, FERPA, and any other applicable regulations and restrictions.'},
     {'title': 'Stored materials are downloadable but not editable, and they are backed up to assure no loss of data. All uploads are approved by a curator before they are published.'}]},
   {'title': 'Derivative Products',
    'subsections': [{'title': 'All materials will contai

# Part 6 — Print Qwen output clearly

In [83]:
for page in qwen_results:
    print("\nPAGE:", page.get("page"))

    if "error" in page:
        print("ERROR:", page["error"])
        print(page.get("raw_response", "")[:1000])

    for item in page.get("items", []):
        print(item.get("label"), "→", item.get("text"))


PAGE: 1

PAGE: 2


# Part 7 — Convert Qwen output to structured blocks

In [84]:
blocks = save_pdfplumber_outputs(pdf_path)
structured_blocks = detect_structure(blocks)

print("Rule-based structural labels:")

for block in structured_blocks:
    if block["label"] in ["document_title", "section", "subsection", "question"]:
        print(block["label"], "→", block["text"])

[2026-05-08 14:16:16] Extracting line-level text with pdfplumber: sample10.pdf
[2026-05-08 14:16:16] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample10.json
[2026-05-08 14:16:16] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample10.txt
Rule-based structural labels:
document_title → DATA MANAGEMENT
section → 1. Policy and Practice
section → 2. Scope
section → 3. Data and Metadata Format and Contents
section → 4. Accessibility and Data Protection
section → 5. Derivative Products
section → 6. Archiving and Access


# Part 8 — Convert Qwen output to structured blocks

In [85]:
qwen_structured_blocks = save_qwen_structured_blocks(
    qwen_output_path=qwen_output_path,
    output_path=qwen_structured_path,
    source_pdf=pdf_path.name
)

print("Saved Qwen structured blocks:", qwen_structured_path.exists())
print("Number of Qwen structured blocks:", len(qwen_structured_blocks))

qwen_structured_blocks[:10]

[2026-05-08 14:16:16] Saved Qwen structured blocks: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample10_structured_blocks.json
Saved Qwen structured blocks: True
Number of Qwen structured blocks: 6


[{'source_pdf': 'sample10.pdf',
  'page': 1,
  'line_order': 1,
  'text': 'Policy and Practice',
  'label': 'section',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample10.pdf',
  'page': 1,
  'line_order': 2,
  'text': 'Scope',
  'label': 'subsection',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample10.pdf',
  'page': 1,
  'line_order': 3,
  'text': 'Data and Metadata Format and Contents',
  'label': 'subsection',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample10.pdf',
  'page': 2,
  'line_order': 4,
  'text': 'Accessibility and Data Protection',
  'label': 'section',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample10.pdf',
  'page': 2,
  'line_order': 5,
  'text': 'Derivative Products',
  'label': 'section',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample10.pdf',
  'page': 2,
  'line_order': 6,
  'text': 'Archiving a

# Part 9 — Build narrative JSON from Qwen blocks

In [86]:
qwen_json = save_narrative_json(
    structured_blocks=qwen_structured_blocks,
    output_path=qwen_json_path,
    skeleton_path=skeleton_path
)

print("Saved Qwen narrative JSON:", qwen_json_path.exists())
print("Qwen JSON path:", qwen_json_path)

[2026-05-08 14:16:16] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample10_qwen.json
Saved Qwen narrative JSON: True
Qwen JSON path: c:\Users\Nahid\dmpbridge\data\structure_json\sample10_qwen.json


# Part 10 — Inspect Qwen narrative JSON

In [87]:
sections = qwen_json["narrative"]["template"]["section"]

print("Number of sections:", len(sections))

for section in sections:
    print(section["order"], section["title"], "| questions:", len(section["question"]))

Number of sections: 4
1 Policy and Practice | questions: 2
2 Accessibility and Data Protection | questions: 0
3 Derivative Products | questions: 0
4 Archiving and Access | questions: 0


In [88]:
sections[0] if sections else "No sections created"

{'id': 'section_1',
 'title': 'Policy and Practice',
 'description': None,
 'order': 1,
 'question': [{'id': 'question_1_1',
   'text': 'Scope',
   'order': 1,
   'answer': {'id': 'answer_1_1',
    'json': {'type': 'text',
     'answer': [{'text': ''}],
     'meta': {'schemaVersion': None}}}},
  {'id': 'question_1_2',
   'text': 'Data and Metadata Format and Contents',
   'order': 2,
   'answer': {'id': 'answer_1_2',
    'json': {'type': 'text',
     'answer': [{'text': ''}],
     'meta': {'schemaVersion': None}}}}]}